In [1]:
import argparse
import os
import pathlib
import sys
from datetime import datetime

import numpy as np
import pandas as pd
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from pycytominer import normalize

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
)
profile_base_dir = root_dir

In [2]:
threshold = 1e5

In [3]:
patient_ids_file_path = pathlib.Path(f"{root_dir}/data/patient_IDs.txt").resolve(
    strict=True
)
patient_ids = pd.read_csv(patient_ids_file_path, header=None).iloc[:, 0].tolist()

In [4]:
# set up log file
log_dir = pathlib.Path("../logs/profile_validation").resolve()
log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir / f"profile_check_{datetime.now():%Y%m%d_%H}.log"


def log(msg, log_file):
    """Print to stdout and write to the log file."""
    print(msg)
    log_file.write(msg + "\n")


with open(log_path, "w") as log_file:
    for patient in patient_ids:
        # construct the profile path dict for this patient
        profile_path_dict = {
            "sc_normalized": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/5.normalized_profiles/sc_norm.parquet"
            ).resolve(strict=True),
            "organoid_normalized": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/5.normalized_profiles/organoid_norm.parquet"
            ).resolve(strict=True),
            "sc_sammed_normalized": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/5.normalized_profiles/sammed_sc_norm.parquet"
            ).resolve(strict=True),
            "organoid_sammed_normalized": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/5.normalized_profiles/sammed_organoid_norm.parquet"
            ).resolve(strict=True),
            "nucleocentric_sammed_normalized": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/5.normalized_profiles/sammed_nucleocentric_norm.parquet"
            ).resolve(strict=True),
            "nucleocentric_morphem_normalized": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/5.normalized_profiles/nucleocentric_morphem_norm.parquet"
            ).resolve(strict=True),
            "sc_fs": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/6.feature_selected_profiles/sc_fs.parquet"
            ).resolve(strict=True),
            "organoid_fs": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/6.feature_selected_profiles/organoid_fs.parquet"
            ).resolve(strict=True),
            "sc_sammed_fs": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/6.feature_selected_profiles/sammed_sc_fs.parquet"
            ).resolve(strict=True),
            "organoid_sammed_fs": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/6.feature_selected_profiles/sammed_organoid_fs.parquet"
            ).resolve(strict=True),
            "nucleocentric_sammed_fs": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/6.feature_selected_profiles/sammed_nucleocentric_fs.parquet"
            ).resolve(strict=True),
            "nucleocentric_morphem_fs": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/6.feature_selected_profiles/nucleocentric_morphem_fs.parquet"
            ).resolve(strict=True),
            "sc_agg_well": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/7.aggregated_profiles/sc_agg_well_level.parquet"
            ).resolve(strict=True),
            "organoid_agg_well": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/7.aggregated_profiles/organoid_agg_well_level.parquet"
            ).resolve(strict=True),
            "organoid_sammed_agg_well": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/7.aggregated_profiles/sammed_organoid_agg_well_level.parquet"
            ).resolve(strict=True),
            "sc_sammed_agg_well": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/7.aggregated_profiles/sammed_sc_agg_well_level.parquet"
            ).resolve(strict=True),
            "nucleocentric_sammed_agg_well": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/7.aggregated_profiles/sammed_nucleocentric_agg_well_level.parquet"
            ).resolve(strict=True),
            "nucleocentric_morphem_agg_well": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/7.aggregated_profiles/nucleocentric_morphem_agg_well_level.parquet"
            ).resolve(strict=True),
            "sc_agg_treatment": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/8.consensus_profiles/sc_consensus.parquet"
            ).resolve(strict=True),
            "organoid_agg_treatment": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/8.consensus_profiles/organoid_consensus.parquet"
            ).resolve(strict=True),
            "organoid_sammed_agg_treatment": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/8.consensus_profiles/sammed_organoid_consensus.parquet"
            ).resolve(strict=True),
            "sc_sammed_agg_treatment": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/8.consensus_profiles/sammed_sc_consensus.parquet"
            ).resolve(strict=True),
            "nucleocentric_sammed_agg_treatment": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/8.consensus_profiles/sammed_nucleocentric_consensus.parquet"
            ).resolve(strict=True),
            "nucleocentric_morphem_agg_treatment": pathlib.Path(
                f"{profile_base_dir}/data/{patient}/image_based_profiles/8.consensus_profiles/nucleocentric_morphem_consensus.parquet"
            ).resolve(strict=True),
        }

        log(f"\n{'=' * 60}", log_file)
        log(f"Processing patient: {patient}", log_file)
        log(f"{'=' * 60}", log_file)
        log(
            f"{'name':40} | {'shape':>15} | {'nans':>8} | {'infs':>8} | {'dupes':>8} | {'values_above_threshold':>8}",
            log_file,
        )
        log(
            f"{'-' * 40} | {'-' * 15} | {'-' * 8} | {'-' * 8} | {'-' * 8} | {'-' * 8}",
            log_file,
        )

        for name, path in profile_path_dict.items():
            df = pd.read_parquet(path)
            nas = df.isna().sum().sum()
            infs = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
            duplicates = df.duplicated().sum()
            values_above_threshold = (
                (df.select_dtypes(include=[np.number]) > threshold).sum().sum()
            )
            shape = df.shape

            log(
                f"{name:40} | {str(shape):>15} | {nas:>8} | {infs:>8} | {duplicates:>8} | {values_above_threshold:>8}",
                log_file,
            )

print(f"\nLog written to: {log_path}")


Processing patient: NF0014_T1
name                                     |           shape |     nans |     infs |    dupes | values_above_threshold
---------------------------------------- | --------------- | -------- | -------- | -------- | --------
sc_normalized                            |    (2132, 2656) |    80850 |        0 |        0 |     3558
organoid_normalized                      |      (232, 896) |     2568 |        0 |        0 |      460
sc_sammed_normalized                     |    (2645, 9274) |     2719 |        0 |        0 |        0
organoid_sammed_normalized               |     (294, 3102) |        0 |        0 |        0 |        0
nucleocentric_sammed_normalized          |    (2646, 1560) |        7 |        0 |        0 |        0
nucleocentric_morphem_normalized         |    (2646, 1560) |        7 |        0 |        0 |        0
sc_fs                                    |     (2132, 352) |     3037 |        0 |        0 |      383
organoid_fs                 

In [8]:
combined_patient_profiles_path = pathlib.Path(
    f"{profile_base_dir}/data/all_patient_profiles/"
).resolve(strict=True)
with open(log_path, "a") as log_file:
    log("= " * 30, log_file)
    log("Processing combined patient profiles", log_file)
    log("= " * 30, log_file)
    # log(f"\n{'=' * 60}", log_file)
    # log(f"Processing patient: {name}", log_file)
    # log(f"{'=' * 60}", log_file)
    log(
        f"{'name':40} | {'shape':>15} | {'nans':>8} | {'infs':>8} | {'dupes':>8} | {'values_above_threshold':>8}",
        log_file,
    )
    log(
        f"{'-' * 40} | {'-' * 15} | {'-' * 8} | {'-' * 8} | {'-' * 8} | {'-' * 8}",
        log_file,
    )

    for profile_level in combined_patient_profiles_path.iterdir():
        for profile_file in profile_level.glob("*.parquet"):
            name = profile_file.stem

            df = pd.read_parquet(profile_file)
            nas = df.isna().sum().sum()
            infs = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
            duplicates = df.duplicated().sum()
            values_above_threshold = (
                (df.select_dtypes(include=[np.number]) > threshold).sum().sum()
            )
            shape = df.shape

            log(
                f"{name:40} | {str(shape):>15} | {nas:>8} | {infs:>8} | {duplicates:>8} | {values_above_threshold:>8}",
                log_file,
            )

= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 
Processing combined patient profiles
= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 
name                                     |           shape |     nans |     infs |    dupes | values_above_threshold
---------------------------------------- | --------------- | -------- | -------- | -------- | --------
nucleocentric_morphem_norm_fs_profiles   |   (76672, 1560) |     2282 |        0 |        0 |        0
sammed_nucleocentric_norm_fs_profiles    |   (76748, 1559) |     2282 |        0 |        0 |        0
sc_norm_fs_profiles                      |   (67551, 2299) |   265018 |        0 |        0 | 16549803
sammed_sc_norm_fs_profiles               |   (76445, 6976) |   127431 |        0 |        0 |        0
organoid_norm_fs_profiles                |    (20541, 867) |    28630 |        0 |        0 |   759470
sammed_organoid_norm_fs_profiles         |   (23985, 2623) |      180 |        0 |        0 |        0
nuc